# 00 · Build the dataset from the raw files

Goal: produce `data/satellite_and_WOA13_1deg_monthly.nc` — one file, twelve months, seven MODIS inputs and three WOA nutrients on the same 1° grid. `configs/default.yaml` already points at it, so once this notebook has run, `01` and `02` work without any further setup.

Three steps:

1. Download WOA13 nutrient files (NOAA, no login).
2. Download MODIS L3m climatology files (NASA, needs a free Earthdata login — one-time setup below).
3. Regrid MODIS from 9 km to 1° and combine with WOA.

**Already have the raw files?** Put them in `data/raw/` (layout in `data/README.md`) and jump to step 3.

### One-time setup for step 2

1. Create a free NASA Earthdata account at https://urs.earthdata.nasa.gov.
2. In Git Bash (Windows) or a terminal (Mac/Linux), run this once, with your own username and password:

   ```bash
   printf 'machine urs.earthdata.nasa.gov\n    login YOUR_USERNAME\n    password YOUR_PASSWORD\n' > ~/_netrc
   ```

   (On Mac/Linux the file is `~/.netrc`.) That is all — the download code reads it automatically.

In [ ]:
from pathlib import Path
from modis_nutrients.config import load_config
from modis_nutrients.download import download_woa, download_modis
from modis_nutrients.regrid import build_dataset

cfg = load_config()
raw = Path(cfg["data"]["raw_dir"])
print("raw files go to:", raw)

## Step 1 — WOA13 (3 nutrients × 12 months, ~350 MB)

Monthly climatology, 1°, all depth levels. Files already present are skipped.

In [ ]:
download_woa(raw / "woa13", months=range(1, 13));

## Step 2 — MODIS (7 products × 12 months, ~2.5 GB)

Each file is a **monthly climatology**: e.g. `AQUA_MODIS.20030101_20230131.L3m.MC.CHL...` is every January from 2003 to 2023 averaged into one map. `L3m` = Level-3 mapped, 9 km.

If a download returns 404, NASA has re-stamped the file names after reprocessing: run `from modis_nutrients.download import search_modis_files; names = search_modis_files()` and pass `filenames=names` below.

In [ ]:
download_modis(raw / "modis", months=range(1, 13));

## Step 3 — Regrid and combine

`build_dataset` does four things:

1. **Regrid MODIS 9 km → 1°.** The 9 km grid is exactly 12 × 12 pixels per 1° cell, so each 1° value is the plain mean of its 144 pixels (geometric mean for CHL, APH, PIC, POC, which are log-normal). Cells with fewer than 25 % valid pixels become NaN. The workshop notebook instead picked roughly one pixel per cell by interpolation, so results on this file differ slightly from the README numbers — expected.
2. **Extract the WOA surface level** (depth = 0) from each monthly file, keeping the analysed field (`nitrate`) plus the per-cell observed mean (`nitrate_mn`), observation count (`nitrate_dd`) and standard error (`nitrate_se`) for later experiments.
3. **Align** the two grids (both are cell centres at −89.5…89.5 / −179.5…179.5) and merge.
4. **Write** the result as one compressed netCDF with a `month` dimension.

Takes a few minutes; the printout shows valid cells per product and month.

In [ ]:
ds = build_dataset(raw / "modis", raw / "woa13", cfg["data"]["built_path"],
                   products=cfg["data"]["features"],
                   geometric_products=cfg["data"]["log_features"],
                   months=range(1, 13))
ds

Done. `configs/default.yaml` points at this file with `months: [1]`; open `02_results.ipynb` to train and evaluate, and change `cfg["data"]["months"]` there to use other months.